# Idea 5: Climate Robustness of Operating Rules

This notebook analyzes the climate-robustness experiment for the Merced and Tuolumne basins. It compares existing operating rules under historical Livneh hydrology, GCM ACCESS1_0_rcp85 hydrology, and LOCA2 ACCESS_CM2 hydrology. The notebook discovers model results, loads Pywr output CSVs, calculates annual robustness metrics, generates figures, and saves summary tables and PNG figures.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root():
    """Find the repo root from Colab, local Jupyter, or the tutorial folder."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.extend([
        Path("/content/cen_sierra_pywr_new"),
        Path("/content/cen-sierra-pywr"),
    ])

    for candidate in candidates:
        if (candidate / "main.py").exists() and (candidate / "data").exists():
            return candidate

    content = Path("/content")
    if content.exists():
        for candidate in content.glob("*/"):
            if (candidate / "main.py").exists() and (candidate / "data").exists():
                return candidate

    raise FileNotFoundError(
        "Could not find the repo root. In Colab, run: %cd /content/cen_sierra_pywr_new"
    )


PROJECT_ROOT = find_project_root()
RESULTS_ROOT = next((p for p in [PROJECT_ROOT / "RESULTS_", PROJECT_ROOT / "results"] if p.exists()), None)
if RESULTS_ROOT is None:
    raise FileNotFoundError(f"Could not find RESULTS_ or results under repo root: {PROJECT_ROOT}")

FIG_DIR = PROJECT_ROOT / "figures" / "idea5_climate_robustness"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if sns is not None:
    sns.set_theme(style="whitegrid", context="talk")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

PALETTE = {
    "Historical Livneh": "#2F6B4F",
    "GCM ACCESS1-0 RCP8.5": "#B65D2E",
    "LOCA2 ACCESS-CM2": "#3C6EAA",
}

MONTH_ORDER = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
MONTH_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Using results root: {RESULTS_ROOT}")
print(f"Figures will be saved to: {FIG_DIR}")

## Experiment Configuration

The historical runs were executed with the `natural_flow` scenario set, which produces both `Baseline` and `natural_flow` columns. For robustness of the existing operating rules, the notebook uses the `Baseline` columns from those historical outputs.

In [ ]:
HISTORICAL_SCENARIO = "Baseline"

BASINS = {
    "merced": {
        "label": "Merced",
        "data_folder": "Merced_River",
        "terminal_reservoir": "Lake McClure",
        "basin_outflow": "Merced River Outflow",
    },
    "tuolumne": {
        "label": "Tuolumne",
        "data_folder": "Tuolumne_River",
        "terminal_reservoir": "Don Pedro Reservoir",
        "basin_outflow": "Tuolumne River Outflow",
    },
}

SCENARIOS = {
    "Historical Livneh": {
        "result_glob": "Natural Flow*",
        "climate_set": "historical",
        "climate_model": "Livneh",
        "scenario_column": HISTORICAL_SCENARIO,
    },
    "GCM ACCESS1-0 RCP8.5": {
        "result_glob": "GCMs test*",
        "climate_set": "gcms",
        "climate_model": "ACCESS1_0_rcp85",
        "scenario_column": None,
    },
    "LOCA2 ACCESS-CM2": {
        "result_glob": "LOCA2_GCMs test*",
        "climate_set": "LOCA2_gcms",
        "climate_model": "ACCESS_CM2",
        "scenario_column": None,
    },
}

FILES = {
    "storage": "Reservoir_Storage_mcm.csv",
    "elevation": "Reservoir_Elevation_m.csv",
    "hydropower_energy": "Hydropower_Energy_MWh.csv",
    "hydropower_flow": "Hydropower_Flow_mcm.csv",
    "ifr_flow": "InstreamFlowRequirement_Flow_mcm.csv",
    "ifr_min": "InstreamFlowRequirement_Min Flow_mcm.csv",
    "output_flow": "Output_Flow_mcm.csv",
    "output_demand": "Output_Demand_mcm.csv",
    "flood_release": "PiecewiseLink_Flow_mcm.csv",
}


def latest_result_path(basin_key, scenario_label):
    scenario = SCENARIOS[scenario_label]
    matches = sorted(
        RESULTS_ROOT.glob(
            f"{scenario['result_glob']}/{basin_key}/{scenario['climate_set']}/{scenario['climate_model']}"
        ),
        key=lambda p: p.stat().st_mtime,
    )
    if not matches:
        raise FileNotFoundError(f"No results found for {basin_key} / {scenario_label}")
    return matches[-1]


RUNS = {}
for basin_key, basin_cfg in BASINS.items():
    for scenario_label, scenario_cfg in SCENARIOS.items():
        hydrology_path = (
            PROJECT_ROOT
            / "data"
            / basin_cfg["data_folder"]
            / "hydrology"
            / scenario_cfg["climate_set"]
            / scenario_cfg["climate_model"]
            / "preprocessed"
            / "full_natural_flow_daily_mcm.csv"
        )
        RUNS[(basin_key, scenario_label)] = {
            "path": latest_result_path(basin_key, scenario_label),
            "hydrology_path": hydrology_path,
            "scenario_column": scenario_cfg["scenario_column"],
        }

for (basin_key, scenario_label), cfg in RUNS.items():
    print(f"{BASINS[basin_key]['label']:9s} | {scenario_label:24s} | {cfg['path']}")

## Helper Functions

In [ ]:
def water_year(index):
    idx = pd.DatetimeIndex(index)
    return idx.year + (idx.month >= 10).astype(int)


def read_pywr_csv(path, scenario=None):
    """Read flat Pywr CSVs and scenario-expanded Pywr CSVs."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8-sig") as f:
        first_line = f.readline().strip()

    if first_line.startswith("node,"):
        df = pd.read_csv(path, header=[0, 1, 2], index_col=0, parse_dates=True)
        df.index.name = "Date"
        df.columns = pd.MultiIndex.from_tuples(
            [(str(a), str(b), str(c)) for a, b, c in df.columns],
            names=["node", "scenario", "unused"],
        )
        if scenario is None:
            scenario = df.columns.get_level_values("scenario")[0]
        available = sorted(set(df.columns.get_level_values("scenario")))
        if scenario not in available:
            raise ValueError(f"Scenario {scenario!r} not found in {path.name}. Available: {available}")
        df = df.xs(scenario, level="scenario", axis=1, drop_level=False)
        df.columns = df.columns.get_level_values("node")
    else:
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        df.index.name = "Date"

    return df.apply(pd.to_numeric, errors="coerce").sort_index()


def read_run_file(basin_key, scenario_label, filename):
    cfg = RUNS[(basin_key, scenario_label)]
    return read_pywr_csv(cfg["path"] / filename, scenario=cfg["scenario_column"])


def read_hydrology(basin_key, scenario_label):
    cfg = RUNS[(basin_key, scenario_label)]
    df = pd.read_csv(cfg["hydrology_path"], index_col=0, parse_dates=True)
    df.index.name = "Date"
    if "flow" not in df.columns:
        df.columns = ["flow"]
    return df[["flow"]].rename(columns={"flow": scenario_label}).sort_index()


def annual_sum(df):
    out = df.copy()
    out["Water Year"] = water_year(out.index)
    return out.groupby("Water Year").sum(numeric_only=True)


def annual_mean(df):
    out = df.copy()
    out["Water Year"] = water_year(out.index)
    return out.groupby("Water Year").mean(numeric_only=True)


def annual_min(df):
    out = df.copy()
    out["Water Year"] = water_year(out.index)
    return out.groupby("Water Year").min(numeric_only=True)


def seasonal_monthly_profile(series_or_df, agg="mean"):
    df = series_or_df.to_frame() if isinstance(series_or_df, pd.Series) else series_or_df.copy()
    monthly = df.resample("MS").mean()
    monthly["month"] = monthly.index.month
    grouped = monthly.groupby("month")
    profile = grouped.median(numeric_only=True) if agg == "median" else grouped.mean(numeric_only=True)
    return profile.reindex(MONTH_ORDER)


def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")
    return path


def clean_metric_name(name):
    return name.replace("_", " ").replace("mcm", "MCM").replace("mwh", "MWh").title()

## Load Results

In [ ]:
data = {basin_key: {scenario_label: {} for scenario_label in SCENARIOS} for basin_key in BASINS}

for basin_key in BASINS:
    for scenario_label in SCENARIOS:
        sections = data[basin_key][scenario_label]
        for key, filename in FILES.items():
            f = RUNS[(basin_key, scenario_label)]["path"] / filename
            if f.exists():
                sections[key] = read_run_file(basin_key, scenario_label, filename)
        sections["hydrology"] = read_hydrology(basin_key, scenario_label)

for basin_key, scenarios in data.items():
    print(f"\n{BASINS[basin_key]['label']}")
    for scenario_label, sections in scenarios.items():
        first = next(iter(sections.values()))
        print(f"  {scenario_label:24s} | {first.index.min().date()} to {first.index.max().date()}")
        for key, df in sections.items():
            print(f"    {key:18s} {df.shape[0]:6d} rows x {df.shape[1]:2d} cols")

## Calculate Robustness Metrics

Metrics are annualized to make historical and future windows comparable. Delivery reliability is calculated from common demand and delivery output columns. IFR reliability is calculated from instream flow minimum requirements and realized instream-flow recorder values.

In [ ]:
annual_rows = []
summary_rows = []
ifr_node_rows = []

for basin_key, basin_cfg in BASINS.items():
    basin_label = basin_cfg["label"]
    terminal = basin_cfg["terminal_reservoir"]
    outflow_node = basin_cfg["basin_outflow"]

    for scenario_label in SCENARIOS:
        sections = data[basin_key][scenario_label]

        annual_runoff = annual_sum(sections["hydrology"]).iloc[:, 0]
        annual_energy = annual_sum(sections["hydropower_energy"]).sum(axis=1)
        annual_storage_mean = annual_mean(sections["storage"][[terminal]]).iloc[:, 0]
        annual_storage_min = annual_min(sections["storage"][[terminal]]).iloc[:, 0]
        annual_outflow = annual_sum(sections["output_flow"][[outflow_node]]).iloc[:, 0]
        annual_flood = annual_sum(sections["flood_release"]).sum(axis=1) if "flood_release" in sections else pd.Series(dtype=float)

        demand = sections["output_demand"]
        delivery_cols = demand.columns.intersection(sections["output_flow"].columns)
        delivery = sections["output_flow"][delivery_cols]
        demand = demand[delivery_cols]
        shortage = (demand - delivery).clip(lower=0)
        annual_demand = annual_sum(demand).sum(axis=1)
        annual_shortage = annual_sum(shortage).sum(axis=1)
        annual_delivery_reliability = 1 - annual_shortage.divide(annual_demand.replace(0, np.nan))

        ifr_min = sections["ifr_min"]
        ifr_cols = ifr_min.columns.intersection(sections["ifr_flow"].columns)
        ifr_flow = sections["ifr_flow"][ifr_cols]
        ifr_min = ifr_min[ifr_cols]
        ifr_deficit = (ifr_min - ifr_flow).clip(lower=0)
        annual_ifr_req = annual_sum(ifr_min).sum(axis=1)
        annual_ifr_deficit = annual_sum(ifr_deficit).sum(axis=1)
        annual_ifr_reliability = 1 - annual_ifr_deficit.divide(annual_ifr_req.replace(0, np.nan))

        for node in ifr_cols:
            req = ifr_min[[node]]
            deficit = ifr_deficit[[node]]
            total_req = req.sum().iloc[0]
            total_def = deficit.sum().iloc[0]
            ifr_node_rows.append({
                "Basin": basin_label,
                "Scenario": scenario_label,
                "Node": node,
                "IFR reliability": 1 - total_def / total_req if total_req else np.nan,
                "IFR deficit days": int((deficit[node] > 1e-8).sum()),
                "Total IFR deficit (mcm)": total_def,
            })

        annual = pd.DataFrame({
            "Basin": basin_label,
            "Scenario": scenario_label,
            "Water Year": annual_energy.index,
            "runoff_mcm": annual_runoff.reindex(annual_energy.index).values,
            "hydropower_mwh": annual_energy.values,
            "terminal_storage_mean_mcm": annual_storage_mean.reindex(annual_energy.index).values,
            "terminal_storage_min_mcm": annual_storage_min.reindex(annual_energy.index).values,
            "basin_outflow_mcm": annual_outflow.reindex(annual_energy.index).values,
            "delivery_demand_mcm": annual_demand.reindex(annual_energy.index).values,
            "delivery_shortage_mcm": annual_shortage.reindex(annual_energy.index).values,
            "delivery_reliability": annual_delivery_reliability.reindex(annual_energy.index).values,
            "ifr_requirement_mcm": annual_ifr_req.reindex(annual_energy.index).values,
            "ifr_deficit_mcm": annual_ifr_deficit.reindex(annual_energy.index).values,
            "ifr_reliability": annual_ifr_reliability.reindex(annual_energy.index).values,
            "flood_release_proxy_mcm": annual_flood.reindex(annual_energy.index).values if len(annual_flood) else np.nan,
        })
        annual_rows.append(annual)

        summary_rows.append({
            "Basin": basin_label,
            "Scenario": scenario_label,
            "Start": sections["storage"].index.min().date(),
            "End": sections["storage"].index.max().date(),
            "Terminal reservoir": terminal,
            "Mean annual runoff (mcm/yr)": annual_runoff.mean(),
            "Mean annual hydropower (MWh/yr)": annual_energy.mean(),
            "Mean terminal storage (mcm)": sections["storage"][terminal].mean(),
            "Minimum terminal storage (mcm)": sections["storage"][terminal].min(),
            "Mean annual basin outflow (mcm/yr)": annual_outflow.mean(),
            "Mean delivery reliability": annual_delivery_reliability.mean(),
            "Mean annual delivery shortage (mcm/yr)": annual_shortage.mean(),
            "Mean IFR reliability": annual_ifr_reliability.mean(),
            "Mean annual IFR deficit (mcm/yr)": annual_ifr_deficit.mean(),
            "Mean annual flood-release proxy (mcm/yr)": annual_flood.mean() if len(annual_flood) else np.nan,
        })

annual_metrics = pd.concat(annual_rows, ignore_index=True)
summary = pd.DataFrame(summary_rows)
ifr_node_summary = pd.DataFrame(ifr_node_rows)

# Percent change relative to each basin's historical run.
metric_cols = [
    "Mean annual runoff (mcm/yr)",
    "Mean annual hydropower (MWh/yr)",
    "Mean terminal storage (mcm)",
    "Minimum terminal storage (mcm)",
    "Mean annual basin outflow (mcm/yr)",
    "Mean delivery reliability",
    "Mean IFR reliability",
    "Mean annual flood-release proxy (mcm/yr)",
]

change_rows = []
for basin_label, group in summary.groupby("Basin"):
    baseline = group.loc[group["Scenario"] == "Historical Livneh"].iloc[0]
    for _, row in group.iterrows():
        out = {"Basin": basin_label, "Scenario": row["Scenario"]}
        for col in metric_cols:
            denom = baseline[col]
            out[col] = np.nan if pd.isna(denom) or denom == 0 else 100 * (row[col] / denom - 1)
        change_rows.append(out)
percent_change = pd.DataFrame(change_rows)

summary_path = FIG_DIR / "climate_robustness_summary_by_basin.csv"
annual_path = FIG_DIR / "climate_robustness_annual_metrics.csv"
ifr_path = FIG_DIR / "climate_robustness_ifr_node_summary.csv"
summary.to_csv(summary_path, index=False)
annual_metrics.to_csv(annual_path, index=False)
ifr_node_summary.to_csv(ifr_path, index=False)

print(f"Saved summary: {summary_path}")
print(f"Saved annual metrics: {annual_path}")
print(f"Saved IFR node summary: {ifr_path}")
display(summary.round(3))

## Figure 1: Seasonal Hydrologic Forcing

In [ ]:
fig, axes = plt.subplots(1, len(BASINS), figsize=(13, 5), sharey=False)
if len(BASINS) == 1:
    axes = [axes]

for ax, (basin_key, basin_cfg) in zip(axes, BASINS.items()):
    for scenario_label in SCENARIOS:
        profile = seasonal_monthly_profile(data[basin_key][scenario_label]["hydrology"])
        ax.plot(MONTH_LABELS, profile[scenario_label].values, marker="o", linewidth=2.3,
                color=PALETTE[scenario_label], label=scenario_label)
    ax.set_title(f"{basin_cfg['label']} Hydrology")
    ax.set_xlabel("Water-year month")
    ax.set_ylabel("Mean full natural flow (mcm/day)")
    ax.grid(True, alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout()
savefig(fig, "fig01_seasonal_hydrologic_forcing_by_basin.png")
plt.show()

## Figure 2: Terminal Reservoir Seasonal Storage

In [ ]:
fig, axes = plt.subplots(1, len(BASINS), figsize=(13, 5), sharey=False)
if len(BASINS) == 1:
    axes = [axes]

for ax, (basin_key, basin_cfg) in zip(axes, BASINS.items()):
    reservoir = basin_cfg["terminal_reservoir"]
    for scenario_label in SCENARIOS:
        storage = data[basin_key][scenario_label]["storage"][[reservoir]]
        profile = seasonal_monthly_profile(storage)
        ax.plot(MONTH_LABELS, profile[reservoir].values, marker="o", linewidth=2.3,
                color=PALETTE[scenario_label], label=scenario_label)
    ax.set_title(f"{basin_cfg['label']}: {reservoir}")
    ax.set_xlabel("Water-year month")
    ax.set_ylabel("Mean storage (mcm)")
    ax.grid(True, alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout()
savefig(fig, "fig02_terminal_reservoir_storage_by_basin.png")
plt.show()

## Figure 3: Annual Hydropower Generation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
if sns is not None:
    sns.boxplot(data=annual_metrics, x="Basin", y="hydropower_mwh", hue="Scenario",
                palette=PALETTE, width=0.65, fliersize=0, ax=ax)
    sns.stripplot(data=annual_metrics, x="Basin", y="hydropower_mwh", hue="Scenario",
                  palette=PALETTE, dodge=True, size=3, alpha=0.35, ax=ax, legend=False)
else:
    annual_metrics.boxplot(column="hydropower_mwh", by=["Basin", "Scenario"], ax=ax)

ax.set_title("Annual Hydropower Generation")
ax.set_xlabel("")
ax.set_ylabel("Hydropower generation (MWh/year)")
ax.grid(True, axis="y", alpha=0.25)
ax.legend(frameon=False, title="", loc="best")
fig.tight_layout()
savefig(fig, "fig03_annual_hydropower_by_basin.png")
plt.show()

## Figure 4: Delivery Reliability and Shortage

In [ ]:
delivery_summary = summary[[
    "Basin", "Scenario", "Mean delivery reliability", "Mean annual delivery shortage (mcm/yr)"
]].copy()
delivery_summary["Mean delivery reliability (%)"] = 100 * delivery_summary["Mean delivery reliability"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharex=False)
if sns is not None:
    sns.barplot(data=delivery_summary, x="Basin", y="Mean delivery reliability (%)", hue="Scenario",
                palette=PALETTE, ax=axes[0])
    sns.barplot(data=delivery_summary, x="Basin", y="Mean annual delivery shortage (mcm/yr)", hue="Scenario",
                palette=PALETTE, ax=axes[1])
else:
    delivery_summary.pivot(index="Basin", columns="Scenario", values="Mean delivery reliability (%)").plot(kind="bar", ax=axes[0])
    delivery_summary.pivot(index="Basin", columns="Scenario", values="Mean annual delivery shortage (mcm/yr)").plot(kind="bar", ax=axes[1])

axes[0].set_title("Delivery Reliability")
axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Mean reliability (%)")
axes[1].set_title("Delivery Shortage")
axes[1].set_ylabel("Mean shortage (mcm/year)")

for ax in axes:
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False, title="")

fig.tight_layout()
savefig(fig, "fig04_delivery_reliability_and_shortage.png")
plt.show()
display(delivery_summary.round(3))

## Figure 5: Instream Flow Reliability

In [ ]:
ifr_summary = summary[["Basin", "Scenario", "Mean IFR reliability", "Mean annual IFR deficit (mcm/yr)"]].copy()
ifr_summary["Mean IFR reliability (%)"] = 100 * ifr_summary["Mean IFR reliability"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharex=False)
if sns is not None:
    sns.barplot(data=ifr_summary, x="Basin", y="Mean IFR reliability (%)", hue="Scenario",
                palette=PALETTE, ax=axes[0])
    sns.barplot(data=ifr_summary, x="Basin", y="Mean annual IFR deficit (mcm/yr)", hue="Scenario",
                palette=PALETTE, ax=axes[1])
else:
    ifr_summary.pivot(index="Basin", columns="Scenario", values="Mean IFR reliability (%)").plot(kind="bar", ax=axes[0])
    ifr_summary.pivot(index="Basin", columns="Scenario", values="Mean annual IFR deficit (mcm/yr)").plot(kind="bar", ax=axes[1])

axes[0].set_title("IFR Reliability")
axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Mean reliability (%)")
axes[1].set_title("IFR Deficit")
axes[1].set_ylabel("Mean deficit (mcm/year)")

for ax in axes:
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False, title="")

fig.tight_layout()
savefig(fig, "fig05_ifr_reliability_and_deficit.png")
plt.show()
display(ifr_node_summary.round(3))

## Figure 6: Basin Outflow Duration Curves

In [ ]:
fig, axes = plt.subplots(1, len(BASINS), figsize=(13, 5), sharey=False)
if len(BASINS) == 1:
    axes = [axes]

for ax, (basin_key, basin_cfg) in zip(axes, BASINS.items()):
    outflow_node = basin_cfg["basin_outflow"]
    for scenario_label in SCENARIOS:
        series = data[basin_key][scenario_label]["output_flow"][outflow_node].dropna().sort_values(ascending=False).reset_index(drop=True)
        exceedance = 100 * (np.arange(1, len(series) + 1) / (len(series) + 1))
        ax.plot(exceedance, series.values, linewidth=2.2, color=PALETTE[scenario_label], label=scenario_label)
    ax.set_title(f"{basin_cfg['label']} Outflow")
    ax.set_xlabel("Exceedance probability (%)")
    ax.set_ylabel("Daily outflow (mcm/day)")
    ax.set_yscale("symlog", linthresh=0.01)
    ax.grid(True, which="both", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout()
savefig(fig, "fig06_basin_outflow_duration_curves.png")
plt.show()

## Figure 7: Flood-Release Proxy

In [ ]:
flood_summary = annual_metrics[["Basin", "Scenario", "Water Year", "flood_release_proxy_mcm"]].dropna()

fig, ax = plt.subplots(figsize=(11, 5.5))
if sns is not None:
    sns.boxplot(data=flood_summary, x="Basin", y="flood_release_proxy_mcm", hue="Scenario",
                palette=PALETTE, width=0.65, fliersize=0, ax=ax)
    sns.stripplot(data=flood_summary, x="Basin", y="flood_release_proxy_mcm", hue="Scenario",
                  palette=PALETTE, dodge=True, size=3, alpha=0.35, ax=ax, legend=False)
else:
    flood_summary.boxplot(column="flood_release_proxy_mcm", by=["Basin", "Scenario"], ax=ax)

ax.set_title("Annual Flood-Release Proxy")
ax.set_xlabel("")
ax.set_ylabel("Annual release through piecewise/flood links (mcm/year)")
ax.grid(True, axis="y", alpha=0.25)
ax.legend(frameon=False, title="")
fig.tight_layout()
savefig(fig, "fig07_flood_release_proxy_by_basin.png")
plt.show()

## Figure 8: Relative Robustness Heatmap

Values are percent changes relative to the historical Livneh run for each basin. Positive values are not always good: for example, higher flood-release proxy or higher deficits can indicate stress, while higher hydropower, storage, and reliability are generally beneficial.

In [ ]:
heat = percent_change.copy()
heat["Basin / Scenario"] = heat["Basin"] + " | " + heat["Scenario"]
heat = heat.set_index("Basin / Scenario")[metric_cols]
heat = heat.rename(columns={
    "Mean annual runoff (mcm/yr)": "Runoff",
    "Mean annual hydropower (MWh/yr)": "Hydropower",
    "Mean terminal storage (mcm)": "Mean storage",
    "Minimum terminal storage (mcm)": "Min storage",
    "Mean annual basin outflow (mcm/yr)": "Outflow",
    "Mean delivery reliability": "Delivery rel.",
    "Mean IFR reliability": "IFR rel.",
    "Mean annual flood-release proxy (mcm/yr)": "Flood proxy",
})

fig, ax = plt.subplots(figsize=(11.5, 6.2))
if sns is not None:
    sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdBu", center=0, linewidths=0.5,
                cbar_kws={"label": "% change vs historical"}, ax=ax)
else:
    vmax = np.nanmax(np.abs(heat.values))
    im = ax.imshow(heat.values, cmap="RdBu", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(heat.shape[1]), heat.columns, rotation=35, ha="right")
    ax.set_yticks(range(heat.shape[0]), heat.index)
    fig.colorbar(im, ax=ax, label="% change vs historical")

ax.set_title("Climate Robustness Metrics Relative to Historical")
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
savefig(fig, "fig08_relative_robustness_heatmap.png")
plt.show()
display(heat.round(2))

## Auto-Generated Result Summary

The cell below writes a short, data-driven interpretation from the summary table. Use it as a first draft, then revise wording for the report.

In [ ]:
def pct_change(row, baseline, col):
    if pd.isna(row[col]) or pd.isna(baseline[col]) or baseline[col] == 0:
        return np.nan
    return 100 * (row[col] / baseline[col] - 1)

lines = []
for basin_label, group in summary.groupby("Basin"):
    baseline = group.loc[group["Scenario"] == "Historical Livneh"].iloc[0]
    lines.append(f"{basin_label}:")
    for scenario_label in ["GCM ACCESS1-0 RCP8.5", "LOCA2 ACCESS-CM2"]:
        row = group.loc[group["Scenario"] == scenario_label].iloc[0]
        runoff_change = pct_change(row, baseline, "Mean annual runoff (mcm/yr)")
        hydro_change = pct_change(row, baseline, "Mean annual hydropower (MWh/yr)")
        storage_change = pct_change(row, baseline, "Mean terminal storage (mcm)")
        delivery_change = pct_change(row, baseline, "Mean delivery reliability")
        ifr_change = pct_change(row, baseline, "Mean IFR reliability")
        flood_change = pct_change(row, baseline, "Mean annual flood-release proxy (mcm/yr)")
        lines.append(
            f"- {scenario_label}: runoff {runoff_change:+.1f}%, hydropower {hydro_change:+.1f}%, "
            f"mean terminal storage {storage_change:+.1f}%, delivery reliability {delivery_change:+.1f}%, "
            f"IFR reliability {ifr_change:+.1f}%, flood-release proxy {flood_change:+.1f}% relative to historical."
        )
    lines.append("")

print("\n".join(lines))

## Notes for Interpretation

The results should be interpreted as a stress test of existing operating rules, not as an optimized climate-adaptation strategy. Lower runoff, lower terminal storage, reduced hydropower, increased delivery shortages, or increased IFR deficits indicate reduced robustness. Higher flood-release proxy can indicate more water moving through high-flow/flood pathways, while lower flood-release proxy under drier futures may reflect reduced runoff rather than improved operations. Because the historical and future windows differ in length, annualized metrics are preferred over total-period sums.